# Setup


In [ ]:
# Working directory should be the root directory of repository.
# setwd("./")
renv::load()
source("./results/utils.R")

suppressPackageStartupMessages({
  library(CAdir)
  library(APL)

  library(SingleCellExperiment)
  library(scater)
  library(scuttle)
  library(scran)

  library(dplyr)
  library(tidyr)

  library(patchwork)
})

options(repr.plot.width = 20, repr.plot.height = 15)

dir <- "./results/"
imgdir <- file.path(dir, "img/review/marker_genes/")
dir.create(imgdir, recursive = TRUE)

## Load data


In [ ]:
sce <- sce_pbmc3k()

# CAdir


In [ ]:
set.seed(1234)

sce_bu <- sce
sce_var <- scran::modelGeneVar(sce)
sce_top <- scran::getTopHVGs(sce_var, prop = 0.4)
sce <- sce[sce_top, ]
sce <- runUMAP(sce, ntop = 2000)

ca <- cacomp(
  obj = as.matrix(logcounts(sce)),
  princ_coords = 3,
  dims = 20,
  top = nrow(sce),
  residuals = "pearson",
  python = TRUE,
  clip = TRUE
)

## Split & Merge Clustering


In [ ]:
set.seed(1)
cabic <- dirclust_splitmerge(
  caobj = ca,
  k = 9,
  cutoff = NULL,
  method = "random",
  apl_quant = 0.99,
  min_cells = 30,
  make_plots = TRUE,
  apl_cutoff_reps = 100,
  qcutoff = 0.9,
  convergence_thr = 0.001
)

### Annotate clusters


In [ ]:
cabic <- CAdir::annotate_biclustering(
  obj = cabic,
  universe = rownames(sce),
  org = "hs"
)

cabic <- rank_genes(cadir = cabic, caobj = ca)
cabic

topg <- top_genes(cabic)

sce$cadir <- cabic@cell_clusters

um1 <- plotUMAP(sce, colour = "cadir")
um2 <- plotUMAP(sce, colour = "cell_type")

ari <- aricode::clustComp(sce$cadir, sce$cell_type)
p <- um1 + ggtitle(paste0("ARI: ", round(ari$ARI, 2))) + um2
p

ggsave(
  plot = p,
  file = file.path(imgdir, "umap.png"),
  width = 3600,
  height = 1800,
  units = "px"
)

# Plot clusters


In [ ]:
b_cells <- cluster_apl(
  ca,
  cabic,
  cluster = "B_cell",
  direction = cabic@directions["B_cell", ],
  group = which(cabic@cell_clusters == "B_cell"),
  highlight_cluster = TRUE,
  show_genes = TRUE,
  label_genes = TRUE,
  show_lines = FALSE,
  size_factor = 0.5,
  ntop = 10
)

ggsave(
  plot = b_cells,
  file = file.path(imgdir, "b_cells_apl.pdf"),
  width = 1600,
  height = 1600,
  units = "px"
)

## Sankey of clustering

In [ ]:
library(ggsankey)
sce_sank <- sce

data <- colData(sce_sank) %>%
  as.data.frame() %>%
  rename('CAdir' = "cadir", 'Annotation' = "cell_type")

df <- data %>%
  make_long(CAdir, Annotation)

info <- data.frame(
  Biclusters = as.factor(data$CAdir),
  Truth = as.character(data$Annotation)
) %>%
  as_tibble() %>%
  group_by(Biclusters, Truth) %>%
  summarise(n = n()) %>%
  group_by(Truth) %>%
  mutate(Partition = n / sum(n)) %>%
  ungroup() %>%
  as.data.frame() %>%
  group_by(Truth) %>%
  filter(Partition == max(Partition)) %>%
  dplyr::select(Biclusters, Truth) %>%
  distinct(Biclusters, .keep_all = TRUE)


pal <- mpi_extend_pal()

cols <- pal(length(unique(sce_sank$cadir)))
names(cols) <- unique(sce_sank$cadir)

info <- left_join(
  data.frame("Biclusters" = names(cols), "col" = cols),
  info,
  by = "Biclusters"
)

cols <- c(cols, info$col)

names(cols) <- c(as.character(unique(sce_sank$cadir)), info$Truth)

order <- c(
  "cluster_6",
  "Monocyte",
  "CD8+_T_cell",
  "Natural_killer_cell",
  "B_cell",
  "Megakaryocyte",
  "0 - Naive CD4 T",
  "2 - Memory CD4 T",
  "1 - CD14+ Mono",
  "5 - FCGR3A+ Mono",
  "7 - DC",
  "4 - CD8 T",
  "6 - NK",
  "3 - B",
  "8 - Platelet"
)

df$node <- gsub("_", " ", df$node)
df$next_node <- gsub("_", " ", df$next_node)
names(cols) <- gsub("_", " ", names(cols))
order <- gsub("_", " ", order)

wrap <- 17
df$node <- factor(
  stringr::str_wrap(df$node, wrap),
  levels = stringr::str_wrap(order, wrap)
)
df$next_node <- factor(
  stringr::str_wrap(df$next_node, wrap),
  levels = stringr::str_wrap(order, wrap)
)

names(cols) <- stringr::str_wrap(names(cols), wrap)

sank <- ggplot(
  df,
  aes(
    x = x,
    next_x = next_x,
    node = node,
    next_node = next_node,
    fill = node,
    label = as.character(node)
  )
) +
  geom_sankey(
    flow.alpha = 0.4,
  ) +
  geom_sankey_label(size = 6) +
  labs(x = "") +
  scale_fill_manual(values = cols) +
  theme_sankey(base_size = 26, base_family = "sans") +
  theme(
    legend.position = "none"
  )

sank

ggsave(
  plot = sank,
  filename = file.path(imgdir, "sankey_plot_clustering.pdf"),
  width = 2700,
  height = 3200,
  device = cairo_pdf,
  units = "px"
)

ggsave(
  plot = sank,
  filename = file.path(imgdir, "sankey_plot_clustering.png"),
  width = 2700,
  height = 3200,
  units = "px"
)

In [ ]:
top_b <- cabic@gene_ranks$B_cell %>%
  filter(Score > 0)
gs <- CAbiNet:::load_gene_set(set = "CellMarker", org = "hs")
gs_bcells <- gs %>%
  filter(cell_type == "B cell") %>%
  pull(gene) %>%
  unique()

cat("% of B cell marker genes:", mean(top_b$Rowname %in% gs_bcells), "\n")

In [ ]:
top_m <- cabic@gene_ranks$Monocyte %>%
  filter(Score > 0)
gs <- CAbiNet:::load_gene_set(set = "CellMarker", org = "hs")
gs_mono <- gs %>%
  filter(cell_type == "Monocyte") %>%
  pull(gene) %>%
  unique()

cat("% of Monocyte marker genes:", mean(top_m$Rowname %in% gs_mono), "\n")

In [ ]:
top_t <- cabic@gene_ranks$`CD8+_T_cell` %>%
  filter(Score > 0)
gs <- CAbiNet:::load_gene_set(set = "CellMarker", org = "hs")
gs_tcells <- gs %>%
  filter(cell_type == "CD8+ T cell") %>%
  pull(gene) %>%
  unique()

cat("% of CD8+ T cells marker genes:", mean(top_t$Rowname %in% gs_tcells), "\n")

In [ ]:
top_t <- cabic@gene_ranks$`CD8+_T_cell` %>%
  filter(Score > 0)
gs <- CAbiNet:::load_gene_set(set = "CellMarker", org = "hs")
gs_tcells <- gs %>%
  filter(cell_type == "T cell") %>%
  pull(gene) %>%
  unique()

cat("% of T cells marker genes:", mean(top_t$Rowname %in% gs_tcells), "\n")

In [ ]:
top_nk <- cabic@gene_ranks$Natural_killer_cell %>%
  filter(Score > 0)

gs <- CAbiNet:::load_gene_set(set = "CellMarker", org = "hs")
gs_nkcells <- gs %>%
  filter(cell_type == "Natural killer cell") %>%
  pull(gene) %>%
  unique()

cat("% of NK cells marker genes:", mean(top_nk$Rowname %in% gs_nkcells), "\n")

In [ ]:
top_mega <- cabic@gene_ranks$Megakaryocyte %>%
  filter(Score > 0)

gs <- CAbiNet:::load_gene_set(set = "CellMarker", org = "hs")
gs_mkcells <- gs %>%
  filter(cell_type == "Megakaryocyte") %>%
  pull(gene) %>%
  unique()

cat(
  "% of Megakaryocyte marker genes:",
  mean(top_mega$Rowname %in% gs_mkcells),
  "\n"
)

In [ ]:
top_c6 <- cabic@gene_ranks$cluster_6

gs <- CAbiNet:::load_gene_set(set = "CellMarker", org = "hs")
gs_cd4tcells <- gs %>%
  filter(cell_type == "CD4+ T cell") %>%
  pull(gene) %>%
  unique()

cat("% of cd4 marker genes:", mean(top_c6$Rowname %in% gs_cd4tcells), "\n")

In [ ]:
top_mega <- cabic@gene_ranks$Megakaryocyte %>%
  filter(Score > 0)

gs <- CAbiNet:::load_gene_set(set = "CellMarker", org = "hs")
gs_plate <- gs %>%
  filter(cell_type == "Platelet") %>%
  pull(gene) %>%
  unique()

cat("% of Platelet marker genes:", mean(top_mega$Rowname %in% gs_plate), "\n")

In [ ]:
unique(gs$cell_type)[grep("T cell", unique(gs$cell_type))]

In [ ]:
cm <- CAbiNet::cellmarker_v2
cm <- cm[cm$species == "Human", ]
cm <- cm[cm$cell_name == "B cell", ]

# Pie charts


In [ ]:
perc1 <- mean(top_b$Rowname %in% gs_bcells)
# pie(c(perc1, 1 - perc1), labels = c("Markers", "Other"), col = rev(CAdir::mpimg_pal()(2)))

perc2 <- mean(top_t$Rowname %in% gs_tcells)
# pie(c(perc2, 1 - perc2), labels = c("Markers", "Other"), col = rev(CAdir::mpimg_pal()(2)))

perc3 <- mean(top_m$Rowname %in% gs_mono)
# pie(c(perc3, 1 - perc3), labels = c("Markers", "Other"), col = rev(CAdir::mpimg_pal()(2)))

perc4 <- mean(top_nk$Rowname %in% gs_nkcells)
# pie(c(perc4, 1 - perc4), labels = c("Markers", "Other"), col = rev(CAdir::mpimg_pal()(2)))

perc5 <- mean(top_mega$Rowname %in% gs_mkcells)
# pie(c(perc5, 1 - perc5), labels = c("Markers", "Other"), col = rev(CAdir::mpimg_pal()(2)))

perc6 <- mean(top_c6$Rowname %in% gs_cd4tcells)

In [ ]:
library(gridGraphics)

piechart <- function(perc, title) {
  pie(
    x = c(perc, 1 - perc),
    labels = c("Markers", "Other"),
    col = rev(CAdir::mpimg_pal()(2))
  )
  title(main = title, cex.main = 1.2, line = 0.1)
}
pie1 <- wrap_elements(full = ~ piechart(perc1, "B cells"))
pie2 <- wrap_elements(full = ~ piechart(perc2, "CD8+ T cells"))
pie3 <- wrap_elements(full = ~ piechart(perc3, "Monocytes"))
pie4 <- wrap_elements(full = ~ piechart(perc4, "NK cells"))
pie5 <- wrap_elements(full = ~ piechart(perc5, "Megakaryocytes"))
pie6 <- wrap_elements(full = ~ piechart(perc6, "cluster 6\n(CD4+ T cells)"))

pdf(file = file.path(imgdir, "pies.pdf"), width = 9, height = 6)
(pie1 + pie2 + pie3 + pie4 + pie5) +
  plot_layout(ncol = 3) +
  plot_annotation(tag_levels = "a")
dev.off()

png(
  file = file.path(imgdir, "pies.png"),
  width = 900,
  height = 600,
  units = "px"
)
(pie1 + pie2 + pie3 + pie4 + pie5) +
  plot_layout(ncol = 3) +
  plot_annotation(tag_levels = "a")
dev.off()

# Pie + Sankey

In [ ]:
# pdf(file = file.path(imgdir, "pies_and_sankey.pdf"), width = 12, height = 6)
# ((pie1 + pie2 + pie3 + pie4 + pie5) | sank) +
#   plot_annotation(tag_levels = "a") &
#   theme(plot.tag = element_text(size = 16))
# dev.off()

png(
  file = file.path(imgdir, "pies_and_sankey.png"),
  width = 1200,
  height = 600,
  units = "px"
)
((pie1 + pie2 + pie3 + pie4 + pie5 + pie6) | sank) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(size = 16))
dev.off()

# B cell discussion

In [ ]:
top_b
top_m

# Top genes table


In [ ]:
library(xtable)
tbl <- top_genes(cabic, cutoff = 0)

top_table <- list()
n <- 3
for (c in unique(tbl$Cluster)) {
  cat(c)
  sub_tbl <- tbl[tbl$Cluster == c, ] %>%
    dplyr::arrange(desc(Score))
  top_table[[c]] <- sub_tbl[seq(n), ]
}
top_table <- do.call("rbind", top_table)
top_table <- top_table[, c(1, 2, 4)]
colnames(top_table) <- c("Gene", "Score", "Cluster")
top_table <- top_table[, c("Cluster", "Gene", "Score")]
rownames(top_table) <- NULL
print(
  xtable(top_table, type = "latex"),
  file = file.path(imgdir, "top_genes.tex")
)

# Session Info

In [ ]:
sessionInfo()